# Question B: baseline AC model

IS4001 Smart Building Analytics project.

**What this script does**

For each air-conditioned zone on Floors 2-7, it trains a model that predicts **normal hourly AC
power** from weather, time and internal heat gains, then checks whether the model is accurate
enough to use as a baseline.

1. Loads the clean hourly table made by `cubems_prepare` (weather and holidays already included).
2. Keeps occupied workday hours only (`is_occupied`), and drops hours that are not normal operation:
   whole-floor AC off on a workday (closure or metering outage) and stuck meter readings.
3. Builds the inputs:
   - **weather**: outdoor temperature, humidity and solar radiation, plus lagged weather
     (temperature 2 h earlier, mean of the previous 24 h, solar 1 h earlier) because the building
     stores heat;
   - **internal gains**: the zone's lighting and plug power, a proxy for occupancy and equipment heat.
     They are not caused by the AC, and a setpoint change does not move them;
   - **time**: hour, day of week, day after a holiday, and the first two hours after a weekend or
     holiday (the morning pull-down).
4. Lets the temperature effect vary: a separate slope for morning, midday and afternoon, and a
   **cooling change point** (ASHRAE change-point model) above which each extra °C costs more.
5. Splits time-wise: **train Jul 2018 - Jun 2019**, **test Jul - Dec 2019**. The last three training
   months (Apr - Jun 2019) are a **validation** window: every modelling choice is made there, so the
   test period is scored once and never used to choose.
6. Looks for a **step change** in each zone's use (a date after which the zone runs at a different
   level for the same weather). On the validation window it compares three ways of handling drift:
   none, a regime indicator from the detected date, and weighting recent months more.
7. Fits a linear regression (primary, statsmodels) and gradient boosting (challenger) and picks one
   per zone on the validation window.
8. Scores the chosen model on the test period with **ASHRAE Guideline 14**: hourly CV(RMSE) <= 30 %
   and NMBE within +/-10 %, and monthly CV(RMSE) <= 15 % and NMBE within +/-5 %. It also reports a
   **rolling-origin** check (retrain on the previous 12 months, predict the next month) and the
   **fractional savings uncertainty** of a 10 % saving.
9. Reports the **AC kW per +1 °C outdoors** with Newey-West (HAC) standard errors, because hourly
   residuals are strongly autocorrelated and ordinary standard errors overstate the certainty.
10. Saves predictions and residuals (actual - predicted) for every occupied hour, ready for the EWMA
    anomaly flags and the comfort cross-check, plus coefficients, metrics and charts.

**Important:** the baseline describes how the building *normally* ran, overcooling included.
It is not a target. It answers "how much AC would this hour normally use, given the weather and
how busy the zone was?", so savings and anomalies are measured against business as usual.

**Requirements:** `pip install pandas numpy scipy statsmodels scikit-learn matplotlib joblib`
(optional: `lightgbm`).

## 1. Settings

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
import joblib

DATA_DIR = Path("../../data/qns2/processed")          # OUT_DIR of cubems_prepare
OUT_DIR = Path("../../data/qns2/baseline")
FLOORS = [2, 3, 4, 5, 6, 7]                # floors with zone sensors (Floor 1 has none)

TRAIN_START, TRAIN_END = "2018-07-01", "2019-06-30"
VALID_START = "2019-04-01"                 # last 3 training months: used to choose, never to score
TEST_START, TEST_END = "2019-07-01", "2019-12-31"

# Inputs. Indoor temperature is deliberately NOT an input: it is mostly caused by the AC
# itself, so using it would make the model circular (and it is missing Sep 2018 - Mar 2019).
WEATHER_FEATURES = ["out_temp_c", "out_rh_pct", "out_solar_wm2"]
LAG_FEATURES = ["temp_lag2", "temp_roll24", "solar_lag1"]
GAIN_FEATURES = ["light_kw", "plug_kw"]


def linear_formula(regime: bool = False) -> str:
    """One temperature slope per part of the day, plus extra slope above the change point."""
    f = ("ac_kw ~ C(daypart):out_temp_c + cooling_hinge + temp_lag2 + temp_roll24"
         " + out_rh_pct + out_solar_wm2 + solar_lag1 + light_kw + plug_kw"
         " + C(hour) + C(day_of_week) + day_after_holiday + startup_after_break")
    return f + " + regime" if regime else f


CHALLENGER_FEATURES = (WEATHER_FEATURES + LAG_FEATURES + GAIN_FEATURES
                       + ["hour", "day_of_week", "day_after_holiday", "startup_after_break"])

# Morning 8-10, midday 11-13, afternoon 14-16 (office hours are 08:00-16:59)
DAYPART_BINS, DAYPART_LABELS = [-1, 10, 13, 23], ["morning", "midday", "afternoon"]
# The cooling change point is searched between these percentiles of training outdoor temperature.
CHANGE_POINT_PCT, CHANGE_POINT_STEP = (10, 90), 0.5

# ASHRAE Guideline 14 calibration criteria
CVRMSE_LIMIT, NMBE_LIMIT = 30.0, 10.0                  # hourly
MONTHLY_CVRMSE_LIMIT, MONTHLY_NMBE_LIMIT = 15.0, 5.0   # monthly
# Fractional savings uncertainty of an assumed saving; ASHRAE 14 asks for < 50 % at 68 % confidence.
ASSUMED_SAVINGS, FSU_CONFIDENCE, FSU_LIMIT = 0.10, 0.68, 50.0

# Model choice on the validation window: score = CV(RMSE) + |NMBE|. The challenger replaces linear
# regression only if it passes ASHRAE 14 there and beats the best linear score by this many points.
CHALLENGER_MARGIN = 2.0

# Step change: a break date is flagged when the mean residual after it differs from before by
# more than this share of mean AC power. Each side needs at least BREAK_MIN_DAYS of data.
STEP_CHANGE_THRESHOLD, BREAK_MIN_DAYS = 0.10, 30
RECENCY_HALF_LIFE_DAYS = 120               # weight of a training hour halves every 120 days

# Newey-West standard errors: allow correlation across about 3 office days of hourly rows.
HAC_MAXLAGS = 27

# Data cleaning
FLOOR_OFF_KW = 1.0     # whole-floor AC below this on a workday hour = closure or metering outage
FLATLINE_HOURS = 4     # the same non-zero reading this many hours in a row = stuck meter

# A zone needs at least this many occupied training hours (about 4 months of workdays).
MIN_TRAIN_HOURS = 800
# Zones whose AC averages below this during occupied hours are skipped (effectively unused).
MIN_MEAN_KW = 0.5
MIN_VALID_HOURS = 150

RANDOM_SEED = 42

## 2. Load data, clean it and build the inputs

In [2]:
def load_floor(floor: int) -> pd.DataFrame:
    path = DATA_DIR / "clean_hourly" / f"floor{floor}_hourly.csv"
    parquet = path.with_suffix(".parquet")
    if parquet.exists():
        df = pd.read_parquet(parquet)
        if "timestamp" in df.columns:
            df = df.set_index("timestamp")
    else:
        df = pd.read_csv(path, parse_dates=["timestamp"], index_col="timestamp")
    # kWh in one hour is numerically the average kW over that hour, which is what the model uses.
    df.columns = df.columns.str.replace(r"_kwh$", "_kw", regex=True)
    for col in ["is_occupied", "is_holiday", "is_workday", "day_after_holiday", "is_weekend"]:
        if col in df:
            df[col] = df[col].astype(str).str.lower().isin(["true", "1"])
    return add_shared_features(df.sort_index())


def add_shared_features(df: pd.DataFrame) -> pd.DataFrame:
    """Lagged weather, part of day and the post-weekend start-up flag (same for every zone)."""
    hours = pd.date_range(df.index.min(), df.index.max(), freq="h")
    temp = df["out_temp_c"].reindex(hours)
    solar = df["out_solar_wm2"].reindex(hours)
    df["temp_lag2"] = temp.shift(2).reindex(df.index)
    df["temp_roll24"] = temp.rolling(24, min_periods=18).mean().shift(1).reindex(df.index)
    df["solar_lag1"] = solar.shift(1).reindex(df.index)
    df["daypart"] = pd.cut(df["hour"], DAYPART_BINS, labels=DAYPART_LABELS).astype(str)

    # First workday after a weekend or holiday; the AC pulls the building down in its first hours.
    days = pd.date_range(df.index.min().normalize(), df.index.max().normalize(), freq="D")
    workday = df.groupby(df.index.normalize())["is_workday"].first().reindex(days, fill_value=False)
    after_break = workday & ~workday.shift(1, fill_value=True)
    df["startup_after_break"] = (after_break.reindex(df.index.normalize()).to_numpy()
                                 & (df["hour"] <= 9).to_numpy()).astype(int)
    return df


def ac_zones(df: pd.DataFrame) -> list[str]:
    return sorted(c.replace("_ac_total_kw", "") for c in df.columns if c.endswith("_ac_total_kw"))


def abnormal_hours(df: pd.DataFrame, zone: str) -> pd.DataFrame:
    """Occupied hours that are not normal operation and should not train or score the baseline."""
    ac = df[f"{zone}_ac_total_kw"]
    run = (ac.diff() != 0).cumsum()
    flat = (ac.groupby(run).transform("size") >= FLATLINE_HOURS) & (ac > 0)
    return pd.DataFrame({"floor_off": df["floor_ac_kw"] < FLOOR_OFF_KW, "flatline": flat})


def zone_frame(df: pd.DataFrame, zone: str) -> tuple[pd.DataFrame, dict]:
    """Occupied normal-operation hours with a valid target and weather, in the model columns."""
    for col in GAIN_FEATURES:
        if f"{zone}_{col}" not in df:
            raise KeyError(f"{zone}: no {zone}_{col} column; the baseline needs internal gains")
    occ = df["is_occupied"]
    bad = abnormal_hours(df, zone)[occ]
    cols = (["hour", "day_of_week", "day_after_holiday", "month", "daypart", "startup_after_break"]
            + WEATHER_FEATURES + LAG_FEATURES)
    z = df.loc[occ, cols].copy()
    z["ac_kw"] = df.loc[occ, f"{zone}_ac_total_kw"]
    for col in GAIN_FEATURES:
        z[col] = df.loc[occ, f"{zone}_{col}"]
    z["day_after_holiday"] = z["day_after_holiday"].astype(int)
    z["split"] = np.select(
        [(z.index >= TRAIN_START) & (z.index <= f"{TRAIN_END} 23:59"),
         (z.index >= TEST_START) & (z.index <= f"{TEST_END} 23:59")],
        ["train", "test"], "other")

    has_data = z["ac_kw"].notna()
    log = {"zone": zone, "occupied_hours": int(occ.sum()), "missing_ac": int((~has_data).sum()),
           "dropped_floor_off": int((bad["floor_off"] & has_data).sum()),
           "dropped_flatline": int((bad["flatline"] & ~bad["floor_off"] & has_data).sum())}
    z = z[~bad["floor_off"] & ~bad["flatline"]]
    z = z.dropna(subset=["ac_kw"] + WEATHER_FEATURES + LAG_FEATURES)
    z = impute_gains(z)
    log.update({"gains_imputed": int(z["gains_imputed"].sum()), "hours_used": len(z)})
    return z, log


def impute_gains(z: pd.DataFrame) -> pd.DataFrame:
    """Fill missing lighting/plug readings with the training median for that hour and weekday."""
    train = z[z["split"] == "train"]
    z["gains_imputed"] = z[GAIN_FEATURES].isna().any(axis=1)
    key = pd.MultiIndex.from_arrays([z["hour"], z["day_of_week"]])
    for col in GAIN_FEATURES:
        typical = train.groupby(["hour", "day_of_week"])[col].median().reindex(key).to_numpy()
        z[col] = z[col].fillna(pd.Series(typical, index=z.index)).fillna(train[col].median())
    return z


def with_change_point(z: pd.DataFrame, tb: float) -> pd.DataFrame:
    z = z.copy()
    z["cooling_hinge"] = (z["out_temp_c"] - tb).clip(lower=0)
    return z

## 3. Accuracy metrics (ASHRAE Guideline 14)
CV(RMSE) measures typical error size relative to average use. NMBE measures whether the model
is biased high or low overall. `p` is the number of model parameters (training fit);
for out-of-sample checks p = 1 is used, as is usual.

The **fractional savings uncertainty** (ASHRAE 14, Annex B) says how precisely a saving of a given
size could be measured against this baseline over the test period's length. It corrects for
autocorrelated residuals through the effective number of independent hours n' = n (1 - ρ) / (1 + ρ).

In [3]:
def ashrae_metrics(actual, predicted, p: int = 1) -> dict:
    y, yhat = np.asarray(actual, float), np.asarray(predicted, float)
    n, mean = len(y), y.mean()
    dof = max(n - p, 1)
    rmse_dof = np.sqrt(np.sum((y - yhat) ** 2) / dof)
    ss_res, ss_tot = np.sum((y - yhat) ** 2), np.sum((y - mean) ** 2)
    return {
        "n_hours": n,
        "mean_kw": round(mean, 3),
        "cv_rmse_pct": round(rmse_dof / mean * 100, 2) if mean else np.nan,
        "nmbe_pct": round(np.sum(y - yhat) / (dof * mean) * 100, 2) if mean else np.nan,
        "r2": round(1 - ss_res / ss_tot, 3) if ss_tot else np.nan,
        "mae_kw": round(np.mean(np.abs(y - yhat)), 3),
    }


def passes_ashrae(m: dict) -> bool:
    return m["cv_rmse_pct"] <= CVRMSE_LIMIT and abs(m["nmbe_pct"]) <= NMBE_LIMIT


def aggregate_metrics(actual: pd.Series, predicted: pd.Series, freq: str) -> dict:
    """CV(RMSE) and NMBE of energy totals per day ("D") or month ("M")."""
    period = actual.index.to_period(freq)
    m = ashrae_metrics(actual.groupby(period).sum(), predicted.groupby(period).sum())
    return {"cv_rmse_pct": m["cv_rmse_pct"], "nmbe_pct": m["nmbe_pct"]}


def lag1_autocorrelation(resid: pd.Series) -> float:
    return float(pd.Series(np.asarray(resid)).autocorr(1))


def savings_uncertainty(cv_rmse_pct: float, n: int, m: int, rho: float, p: int) -> float:
    """ASHRAE 14 fractional savings uncertainty (%) of an ASSUMED_SAVINGS saving over m hours."""
    n_eff = n * (1 - rho) / (1 + rho)
    t = stats.t.ppf(1 - (1 - FSU_CONFIDENCE) / 2, max(n_eff - p, 1))
    u = 1.26 * t * (cv_rmse_pct / 100) * np.sqrt((n / n_eff) * (1 + 2 / n) / m) / ASSUMED_SAVINGS
    return round(u * 100, 1)

## 4. Models

Both models get the same drift handling, chosen per zone on the validation window:
- `none`: every training hour counts equally;
- `regime`: a 0/1 indicator from the detected step-change date (only offered if a step is found);
- `recency`: training hours are weighted down the older they are (half-life 120 days).

Linear regression standard errors are Newey-West (HAC); this changes the p-values, not the predictions.

In [4]:
def make_challenger():
    try:
        from lightgbm import LGBMRegressor
        return "LightGBM", LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=15,
                                         min_child_samples=30, subsample=0.8, subsample_freq=1,
                                         colsample_bytree=0.9, random_state=RANDOM_SEED, verbose=-1)
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingRegressor
        return "HistGradientBoosting", HistGradientBoostingRegressor(
            max_iter=400, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=30,
            random_state=RANDOM_SEED)


CHALLENGER_NAME = make_challenger()[0]


def recency_weights(index: pd.DatetimeIndex) -> np.ndarray:
    age_days = (index.max() - index).days.to_numpy()
    return 0.5 ** (age_days / RECENCY_HALF_LIFE_DAYS)


def fit_linear(data: pd.DataFrame, formula: str, weights=None, robust: bool = True):
    model = smf.ols(formula, data) if weights is None else smf.wls(formula, data, weights=weights)
    return model.fit(cov_type="HAC", cov_kwds={"maxlags": HAC_MAXLAGS}) if robust else model.fit()


def fit_spec(spec: dict, data: pd.DataFrame):
    """spec = {"model": "linear" | "challenger", "drift": "none" | "regime" | "recency"}."""
    weights = recency_weights(data.index) if spec["drift"] == "recency" else None
    if spec["model"] == "linear":
        return fit_linear(data, linear_formula(spec["drift"] == "regime"), weights)
    model = make_challenger()[1]
    model.fit(data[challenger_features(spec)], data["ac_kw"], sample_weight=weights)
    return model


def challenger_features(spec: dict) -> list[str]:
    return CHALLENGER_FEATURES + (["regime"] if spec["drift"] == "regime" else [])


def predict_spec(fitted, spec: dict, data: pd.DataFrame) -> pd.Series:
    if spec["model"] == "linear":
        pred = fitted.predict(data)
    else:
        pred = pd.Series(fitted.predict(data[challenger_features(spec)]), index=data.index)
    return pred.clip(lower=0)


def fit_change_point(data: pd.DataFrame) -> float:
    """Outdoor temperature above which AC rises faster: grid search on training error."""
    lo, hi = np.percentile(data["out_temp_c"], CHANGE_POINT_PCT)
    grid = np.arange(np.floor(lo / CHANGE_POINT_STEP) * CHANGE_POINT_STEP, hi + 1e-9, CHANGE_POINT_STEP)
    ssr = {tb: fit_linear(with_change_point(data, tb), linear_formula(), robust=False).ssr for tb in grid}
    return float(min(ssr, key=ssr.get))


def find_step_change(resid: pd.Series, mean_kw: float) -> dict:
    """The single date that best splits the residuals into two levels (binary segmentation)."""
    daily = resid.groupby(resid.index.normalize()).agg(["sum", "count"])
    s1, n1 = daily["sum"].cumsum().shift(1), daily["count"].cumsum().shift(1)
    total_s, total_n = daily["sum"].sum(), daily["count"].sum()
    m1, m2 = s1 / n1, (total_s - s1) / (total_n - n1)
    score = n1 * (total_n - n1) / total_n * (m2 - m1) ** 2
    days = daily.index
    allowed = (days >= days[0] + pd.Timedelta(days=BREAK_MIN_DAYS)) & \
              (days <= days[-1] - pd.Timedelta(days=BREAK_MIN_DAYS))
    score = score[allowed]
    if score.empty or score.isna().all():
        return {"break_date": None, "shift_kw": 0.0, "shift_share": 0.0, "step_change_flag": False}
    best = score.idxmax()
    shift = float(m2[best] - m1[best])
    return {"break_date": best.date().isoformat(), "shift_kw": round(shift, 3),
            "shift_share": round(shift / mean_kw, 3),
            "step_change_flag": bool(abs(shift / mean_kw) > STEP_CHANGE_THRESHOLD)}


def with_regime(z: pd.DataFrame, break_date) -> pd.DataFrame:
    z = z.copy()
    z["regime"] = (z.index >= pd.Timestamp(break_date)).astype(int) if break_date else 0
    return z


def temperature_effect(fitted, data: pd.DataFrame, formula: str, tb: float) -> dict:
    """Extra AC kW if outdoor temperature were 1 °C higher for every hour (lags included),
    averaged over the given hours, with HAC and ordinary standard errors."""
    warmer = data.copy()
    warmer[["out_temp_c", "temp_lag2", "temp_roll24"]] += 1
    warmer = with_change_point(warmer, tb)
    # Same rows and categories, so both design matrices have the same columns as fitted.params.
    x_base, x_warm = smf.ols(formula, data).exog, smf.ols(formula, warmer).exog
    assert smf.ols(formula, warmer).exog_names == list(fitted.params.index)
    g = (x_warm - x_base).mean(axis=0)
    effect = float(g @ fitted.params.to_numpy())
    se_hac = float(np.sqrt(g @ fitted.cov_params().to_numpy() @ g))
    naive = fitted.model.fit()
    se_ols = float(np.sqrt(g @ naive.cov_params().to_numpy() @ g))
    return {"kw_per_c": round(effect, 3), "se_hac": round(se_hac, 3), "se_ols": round(se_ols, 3),
            "ci95_low": round(effect - 1.96 * se_hac, 3), "ci95_high": round(effect + 1.96 * se_hac, 3),
            "se_inflation": round(se_hac / se_ols, 2) if se_ols else np.nan}

## 5. Choosing a model, and the rolling-origin check

In [5]:
def choose_model(z: pd.DataFrame) -> tuple[dict, dict, float, list[dict]]:
    """Pick model and drift handling on the validation window (Apr - Jun 2019).

    Returns the chosen spec, the step-change check, the change point fitted on the
    fitting window, and the score of every candidate."""
    train = z[z["split"] == "train"]
    fit_part, valid = train[train.index < VALID_START], train[train.index >= VALID_START]
    tb = fit_change_point(fit_part)
    fit_part, valid = with_change_point(fit_part, tb), with_change_point(valid, tb)

    base = fit_linear(fit_part, linear_formula(), robust=False)
    step = find_step_change(fit_part["ac_kw"] - base.fittedvalues, fit_part["ac_kw"].mean())
    if len(valid) < MIN_VALID_HOURS:
        return {"model": "linear", "drift": "none"}, step, tb, []

    fit_part, valid = with_regime(fit_part, step["break_date"]), with_regime(valid, step["break_date"])
    drifts = ["none", "recency"] + (["regime"] if step["step_change_flag"] else [])
    rows = []
    for model in ["linear", "challenger"]:
        for drift in drifts:
            spec = {"model": model, "drift": drift}
            m = ashrae_metrics(valid["ac_kw"], predict_spec(fit_spec(spec, fit_part), spec, valid))
            rows.append({**spec, **m, "score": m["cv_rmse_pct"] + abs(m["nmbe_pct"]),
                         "passes_ashrae14": passes_ashrae(m)})
    best_lin = min((r for r in rows if r["model"] == "linear"), key=lambda r: r["score"])
    best_ch = min((r for r in rows if r["model"] == "challenger"), key=lambda r: r["score"])
    best = best_ch if (best_ch["passes_ashrae14"]
                       and best_ch["score"] <= best_lin["score"] - CHALLENGER_MARGIN) else best_lin
    for r in rows:
        r["chosen"] = r is best
    return {"model": best["model"], "drift": best["drift"]}, step, tb, rows


def rolling_origin(z: pd.DataFrame, spec: dict) -> list[dict]:
    """Retrain on the 12 months before each test month, predict that month.

    This is how a deployed baseline would be maintained (monthly retraining), so it shows
    whether the fixed-model test result is held back by drift."""
    rows = []
    for month in pd.date_range(TEST_START, TEST_END, freq="MS"):
        window = z[(z.index >= month - pd.DateOffset(months=12)) & (z.index < month)]
        target = z[(z.index >= month) & (z.index < month + pd.DateOffset(months=1))]
        if len(window) < MIN_TRAIN_HOURS or len(target) < 20:
            continue
        pred = predict_spec(fit_spec(spec, window), spec, target)
        m = ashrae_metrics(target["ac_kw"], pred)
        rows.append({"month": month.strftime("%Y-%m"), **m, "passes_ashrae14": passes_ashrae(m)})
    return rows

## 6. Charts

In [6]:
def plot_zone(floor: int, zone: str, z: pd.DataFrame, chosen: str, out_dir: Path) -> None:
    test = z[z["split"] == "test"]
    fig, axes = plt.subplots(3, 1, figsize=(11, 10))

    # (a) two sample test weeks: actual vs predicted
    week = test.loc[test.index.min(): test.index.min() + pd.Timedelta(days=13)]
    ax = axes[0]
    ax.plot(week.index, week["ac_kw"], ".", ms=4, label="Actual", color="#444444")
    ax.plot(week.index, week["pred_linear"], ".", ms=4, label="Linear", color="#1f77b4")
    ax.plot(week.index, week["pred_challenger"], ".", ms=4, label="Challenger", color="#ff7f0e", alpha=0.7)
    ax.set_title(f"Floor {floor} {zone}: first two test weeks (occupied hours)")
    ax.set_ylabel("AC power (kW)")
    ax.legend(loc="upper right", frameon=False)

    # (b) actual vs predicted, chosen model, test period
    ax = axes[1]
    pred = test[f"pred_{chosen}"]
    ax.scatter(pred, test["ac_kw"], s=6, alpha=0.4, color="#1f77b4")
    lim = [0, max(pred.max(), test["ac_kw"].max()) * 1.05]
    ax.plot(lim, lim, color="#888888", lw=1)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("Predicted AC power (kW)"); ax.set_ylabel("Actual AC power (kW)")
    ax.set_title(f"Test period, {chosen} model (points on the line = perfect prediction)")

    # (c) mean residual by month, chosen model, whole period
    ax = axes[2]
    monthly = z["resid"].groupby(z.index.to_period("M")).mean()
    def colour(p):
        start = p.to_timestamp()
        return "#ff7f0e" if start >= pd.Timestamp(TEST_START) else \
               "#2ca02c" if start >= pd.Timestamp(VALID_START) else "#1f77b4"
    ax.bar(monthly.index.astype(str), monthly.values, color=[colour(p) for p in monthly.index])
    ax.axhline(0, color="#888888", lw=1)
    ax.set_ylabel("Mean residual (kW)")
    ax.set_title("Actual minus predicted by month (blue = training, green = validation, orange = test)")
    ax.tick_params(axis="x", rotation=60)

    fig.tight_layout()
    fig.savefig(out_dir / f"floor{floor}_{zone}_baseline.png", dpi=130)
    plt.close(fig)

## 7. Run

In [7]:
def run_zone(floor: int, zone: str, df: pd.DataFrame, out: Path) -> dict | None:
    z, clean_log = zone_frame(df, zone)
    train = z[z["split"] == "train"]
    if len(train) < MIN_TRAIN_HOURS or z["ac_kw"].mean() < MIN_MEAN_KW:
        print(f"  {zone}: skipped ({len(train)} training hours, mean {z['ac_kw'].mean():.2f} kW)")
        return None

    # 1. Choose model and drift handling on the validation window
    spec, step, _, candidates = choose_model(z)

    # 2. Refit the change point and the step date on the full training year, then the models
    tb = fit_change_point(train)
    z = with_change_point(z, tb)
    train = z[z["split"] == "train"]
    if spec["drift"] == "regime":
        base = fit_linear(train, linear_formula(), robust=False)
        step["break_date"] = find_step_change(train["ac_kw"] - base.fittedvalues,
                                              train["ac_kw"].mean())["break_date"]
    z = with_regime(z, step["break_date"] if spec["drift"] == "regime" else None)
    train, test = z[z["split"] == "train"], z[z["split"] == "test"]

    linear = fit_spec({"model": "linear", "drift": spec["drift"]}, train)
    challenger = fit_spec({"model": "challenger", "drift": spec["drift"]}, train)
    z["pred_linear"] = predict_spec(linear, {"model": "linear", "drift": spec["drift"]}, z)
    z["pred_challenger"] = predict_spec(challenger, {"model": "challenger", "drift": spec["drift"]}, z)
    chosen = spec["model"]
    z["chosen_model"] = chosen
    z["pred_chosen"] = z[f"pred_{chosen}"]
    z["resid"] = z["ac_kw"] - z["pred_chosen"]
    z["zone"] = zone

    # 3. Score once on the test period
    metrics = []
    for role, col, p in [("linear", "pred_linear", int(linear.df_model) + 1),
                         ("challenger", "pred_challenger", 1)]:
        for split, part in [("train", train), ("test", test)]:
            m = ashrae_metrics(part["ac_kw"], z.loc[part.index, col], p if split == "train" else 1)
            m.update({"zone": zone, "model": CHALLENGER_NAME if role == "challenger" else "Linear regression",
                      "role": role, "split": split, "drift": spec["drift"], "passes_ashrae14": passes_ashrae(m)})
            metrics.append(m)
    test_m = next(m for m in metrics if m["role"] == chosen and m["split"] == "test")
    train_m = next(m for m in metrics if m["role"] == chosen and m["split"] == "train")
    test_rows = z.loc[test.index]
    monthly = aggregate_metrics(test_rows["ac_kw"], test_rows["pred_chosen"], "M")
    daily = aggregate_metrics(test_rows["ac_kw"], test_rows["pred_chosen"], "D")
    rho = lag1_autocorrelation(z.loc[train.index, "resid"])
    p_chosen = int(linear.df_model) + 1
    fsu = savings_uncertainty(train_m["cv_rmse_pct"], len(train), len(test), rho, p_chosen)
    rolling = rolling_origin(z, {"model": chosen, "drift": spec["drift"]})
    temp = temperature_effect(linear, train, linear_formula(spec["drift"] == "regime"), tb)
    # The effect must match a direct check: predict with every hour 1 °C warmer.
    warmer = with_change_point(train.assign(**{c: train[c] + 1 for c in ["out_temp_c", "temp_lag2", "temp_roll24"]}), tb)
    assert abs((linear.predict(warmer) - linear.predict(train)).mean() - temp["kw_per_c"]) < 1e-2

    joblib.dump({"linear": linear, "challenger": challenger, "formula": linear_formula(spec["drift"] == "regime"),
                 "challenger_features": challenger_features(spec), "chosen": chosen, "drift": spec["drift"],
                 "change_point_c": tb, "break_date": step["break_date"]},
                out / "models" / f"floor{floor}_{zone}_baseline.joblib")
    plot_zone(floor, zone, z, chosen, out / "charts")

    lin_t = next(m for m in metrics if m["role"] == "linear" and m["split"] == "test")
    ch_t = next(m for m in metrics if m["role"] == "challenger" and m["split"] == "test")
    status = "PASS" if passes_ashrae(test_m) else "FAIL"
    print(f"  {zone}: linear test CV(RMSE) {lin_t['cv_rmse_pct']:.1f}% NMBE {lin_t['nmbe_pct']:+.1f}% | "
          f"{CHALLENGER_NAME} {ch_t['cv_rmse_pct']:.1f}% / {ch_t['nmbe_pct']:+.1f}% -> using {chosen}, "
          f"drift '{spec['drift']}' (ASHRAE 14 {status})")
    if step["step_change_flag"]:
        print(f"    step change from {step['break_date']}: {step['shift_share']:+.0%} of mean AC power")

    rolling_df = pd.DataFrame(rolling)
    summary = {
        "floor": floor, "zone": zone, "chosen_model": test_m["model"], "test_hours": test_m["n_hours"],
        "mean_kw": test_m["mean_kw"], "cv_rmse_pct": test_m["cv_rmse_pct"], "nmbe_pct": test_m["nmbe_pct"],
        "r2": test_m["r2"], "passes_ashrae14": passes_ashrae(test_m),
        "monthly_cv_rmse_pct": monthly["cv_rmse_pct"], "monthly_nmbe_pct": monthly["nmbe_pct"],
        "passes_ashrae14_monthly": bool(monthly["cv_rmse_pct"] <= MONTHLY_CVRMSE_LIMIT
                                        and abs(monthly["nmbe_pct"]) <= MONTHLY_NMBE_LIMIT),
        "daily_cv_rmse_pct": daily["cv_rmse_pct"], "daily_nmbe_pct": daily["nmbe_pct"],
        "rolling_cv_rmse_pct": round(rolling_df["cv_rmse_pct"].mean(), 2) if rolling else np.nan,
        "rolling_abs_nmbe_pct": round(rolling_df["nmbe_pct"].abs().mean(), 2) if rolling else np.nan,
        "rolling_months_passing": f"{int(rolling_df['passes_ashrae14'].sum())}/{len(rolling_df)}" if rolling else "",
        "fsu_pct": fsu, "resid_lag1_autocorr": round(rho, 3),
        "drift_handling": spec["drift"], "change_point_c": tb,
        "kw_per_c": temp["kw_per_c"], "kw_per_c_se_hac": temp["se_hac"],
    }
    coefs = pd.DataFrame({"coef": linear.params, "std_err_hac": linear.bse, "p_value_hac": linear.pvalues})
    coefs.insert(0, "zone", zone)
    return {"summary": summary, "metrics": metrics, "pred": z, "clean": clean_log,
            "coefs": coefs.reset_index(names="term"),
            "step": {"zone": zone, **step, "drift_chosen": spec["drift"]},
            "candidates": [{"zone": zone, **c} for c in candidates],
            "rolling": [{"zone": zone, **r} for r in rolling],
            "temp": {"zone": zone, "change_point_c": tb, **temp}}


def run_floor(floor: int) -> list[dict]:
    out = OUT_DIR / f"floor{floor}"
    (out / "models").mkdir(parents=True, exist_ok=True)
    (out / "charts").mkdir(parents=True, exist_ok=True)

    df = load_floor(floor)
    zones = ac_zones(df)
    print(f"Floor {floor}: AC zones found {zones}")
    results = [r for r in (run_zone(floor, zone, df, out) for zone in zones) if r is not None]
    if not results:
        print(f"  Floor {floor}: no zone had enough data to train a baseline.")
        return []

    gather = lambda key: [row for r in results for row in (r[key] if isinstance(r[key], list) else [r[key]])]
    metrics = pd.DataFrame(gather("metrics"))[["zone", "model", "role", "split", "drift", "n_hours", "mean_kw",
                                               "cv_rmse_pct", "nmbe_pct", "r2", "mae_kw", "passes_ashrae14"]]
    preds = pd.concat(r["pred"] for r in results)[
        ["zone", "split", "ac_kw", "pred_linear", "pred_challenger", "chosen_model", "pred_chosen", "resid"]
        + WEATHER_FEATURES + GAIN_FEATURES + ["gains_imputed", "hour", "day_of_week", "month"]]
    preds.index.name = "timestamp"

    metrics.to_csv(out / "metrics.csv", index=False)
    pd.concat(r["coefs"] for r in results).to_csv(out / "linear_coefficients.csv", index=False)
    preds.to_csv(out / "predictions_occupied_hours.csv", float_format="%.4f")
    pd.DataFrame(gather("step")).to_csv(out / "step_change_check.csv", index=False)
    pd.DataFrame(gather("candidates")).to_csv(out / "model_selection.csv", index=False)
    pd.DataFrame(gather("rolling")).to_csv(out / "rolling_origin.csv", index=False)
    pd.DataFrame(gather("clean")).to_csv(out / "cleaning_log.csv", index=False)
    pd.DataFrame(gather("temp")).to_csv(out / "temperature_effect.csv", index=False)

    print("  AC power per +1 °C outdoors, sustained (95 % CI with Newey-West errors):")
    for r in results:
        t = r["temp"]
        print(f"    {t['zone']}: {t['kw_per_c']:+.2f} kW/°C [{t['ci95_low']:+.2f}, {t['ci95_high']:+.2f}]; "
              f"HAC error {t['se_inflation']:.1f}x the ordinary one")
    return [r["summary"] for r in results]


def main() -> pd.DataFrame:
    summary = pd.DataFrame([row for floor in FLOORS for row in run_floor(floor)])
    summary.to_csv(OUT_DIR / "summary_metrics.csv", index=False)
    (OUT_DIR / "settings.json").write_text(json.dumps({
        "floors": FLOORS, "train": [TRAIN_START, TRAIN_END], "validation_start": VALID_START,
        "test": [TEST_START, TEST_END], "linear_formula": linear_formula(),
        "challenger": CHALLENGER_NAME, "challenger_features": CHALLENGER_FEATURES,
        "ashrae14_hourly_limits": {"cv_rmse_pct": CVRMSE_LIMIT, "nmbe_pct": NMBE_LIMIT},
        "ashrae14_monthly_limits": {"cv_rmse_pct": MONTHLY_CVRMSE_LIMIT, "nmbe_pct": MONTHLY_NMBE_LIMIT},
        "challenger_margin_score_points": CHALLENGER_MARGIN, "step_change_threshold": STEP_CHANGE_THRESHOLD,
        "recency_half_life_days": RECENCY_HALF_LIFE_DAYS, "hac_maxlags": HAC_MAXLAGS,
        "cleaning": {"floor_off_kw": FLOOR_OFF_KW, "flatline_hours": FLATLINE_HOURS},
        "fsu": {"assumed_savings": ASSUMED_SAVINGS, "confidence": FSU_CONFIDENCE}}, indent=2))

    n = len(summary)
    print(f"\nFloors {FLOORS[0]}-{FLOORS[-1]}: {int(summary['passes_ashrae14'].sum())} of {n} zones pass ASHRAE 14 "
          f"hourly, {int(summary['passes_ashrae14_monthly'].sum())} of {n} pass monthly, "
          f"{int((summary['fsu_pct'] < FSU_LIMIT).sum())} of {n} could measure a {ASSUMED_SAVINGS:.0%} saving "
          f"(FSU < {FSU_LIMIT:g} % at {FSU_CONFIDENCE:.0%} confidence)")
    print(f"Saved to {OUT_DIR.resolve()}")
    return summary


with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    summary = main()
summary

Floor 2: AC zones found ['z1', 'z2', 'z4']


C:\Users\ganad\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\ganad\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\ganad\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ganad\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\ganad\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

  z1: linear test CV(RMSE) 14.0% NMBE +0.7% | HistGradientBoosting 16.3% / -1.7% -> using linear, drift 'none' (ASHRAE 14 PASS)


  z2: linear test CV(RMSE) 22.7% NMBE +13.5% | HistGradientBoosting 23.0% / +12.8% -> using challenger, drift 'recency' (ASHRAE 14 FAIL)
    step change from 2019-01-22: +21% of mean AC power


  z4: linear test CV(RMSE) 21.3% NMBE -4.2% | HistGradientBoosting 22.9% / -1.4% -> using challenger, drift 'regime' (ASHRAE 14 PASS)
    step change from 2018-12-12: -11% of mean AC power


  AC power per +1 °C outdoors, sustained (95 % CI with Newey-West errors):
    z1: +2.17 kW/°C [+1.93, +2.42]; HAC error 1.9x the ordinary one
    z2: +2.46 kW/°C [+1.90, +3.02]; HAC error 3.0x the ordinary one
    z4: +0.36 kW/°C [+0.16, +0.55]; HAC error 3.2x the ordinary one


... (27 outputs left out)

  z4: linear test CV(RMSE) 11.9% NMBE +0.7% | HistGradientBoosting 13.3% / +2.3% -> using linear, drift 'none' (ASHRAE 14 PASS)


  z5: linear test CV(RMSE) 29.2% NMBE +4.2% | HistGradientBoosting 32.3% / -3.9% -> using linear, drift 'recency' (ASHRAE 14 PASS)
    step change from 2018-12-11: -12% of mean AC power


  AC power per +1 °C outdoors, sustained (95 % CI with Newey-West errors):
    z1: +1.76 kW/°C [+1.45, +2.07]; HAC error 2.9x the ordinary one
    z2: +0.59 kW/°C [+0.44, +0.75]; HAC error 1.7x the ordinary one
    z4: +1.17 kW/°C [+0.90, +1.43]; HAC error 2.2x the ordinary one
    z5: +1.54 kW/°C [+1.32, +1.76]; HAC error 2.4x the ordinary one

Floors 2-7: 16 of 23 zones pass ASHRAE 14 hourly, 12 of 23 pass monthly, 23 of 23 could measure a 10% saving (FSU < 50 % at 68% confidence)
Saved to data/qns2/baseline


,floor,zone,chosen_model,test_hours,mean_kw,cv_rmse_pct,nmbe_pct,r2,passes_ashrae14,monthly_cv_rmse_pct,...,daily_nmbe_pct,rolling_cv_rmse_pct,rolling_abs_nmbe_pct,rolling_months_passing,fsu_pct,resid_lag1_autocorr,drift_handling,change_point_c,kw_per_c,kw_per_c_se_hac
0,2,z1,Linear regression,984,27.683,14.01,0.67,0.510,True,8.41,...,0.67,12.89,6.04,5/6,7.2,0.449,none,33.0,2.173,0.125
1,2,z2,HistGradientBoosting,982,28.519,22.99,12.83,-2.163,False,15.03,...,12.92,18.13,8.29,4/6,11.0,0.711,recency,33.0,2.459,0.288
2,2,z4,HistGradientBoosting,984,8.100,22.90,-1.39,-0.032,True,16.60,...,-1.40,19.92,11.32,3/6,3.4,0.411,regime,27.5,0.356,0.100
3,3,z1,Linear regression,976,18.858,17.83,-12.25,0.638,False,14.01,...,-12.34,15.70,10.23,3/6,7.9,0.532,none,29.5,1.444,0.099
4,3,z2,Linear regression,976,15.188,16.49,-4.17,0.724,True,9.39,...,-4.20,15.87,7.48,4/6,6.8,0.417,none,29.5,1.325,0.089
5,3,z4,Linear regression,976,17.912,18.61,13.79,0.698,False,16.03,...,13.89,16.20,9.55,3/6,10.3,0.448,none,27.5,1.558,0.123
6,3,z5,Linear regression,976,7.427,18.52,-1.38,0.747,True,6.36,...,-1.39,17.37,5.34,5/6,10.1,0.403,none,30.0,0.629,0.071
7,4,z1,Linear regression,952,20.968,18.46,2.24,0.641,True,14.38,...,2.26,18.95,12.20,2/6,9.0,0.489,recency,27.5,1.770,0.165
8,4,z2,Linear regression,952,15.806,12.87,7.04,0.725,True,8.37,...,7.09,11.34,4.31,6/6,13.2,0.651,recency,32.5,0.404,0.187
9,4,z4,Linear regression,952,18.726,12.98,0.18,0.720,True,7.87,...,0.18,12.44,5.86,5/6,8.0,0.492,none,31.5,1.332,0.093


## 8. Outputs and next steps

Per floor, in `baseline/floor<N>/`:
- `predictions_occupied_hours.csv`: `resid` (actual minus predicted) for every normal occupied hour.
- `metrics.csv`: train and test accuracy for both models.
- `model_selection.csv`: every candidate's validation score (Apr - Jun 2019) and which was chosen.
- `rolling_origin.csv`: monthly accuracy when retrained on the previous 12 months.
- `step_change_check.csv`: the detected break date and size per zone.
- `temperature_effect.csv`: kW per +1 °C with HAC and ordinary standard errors.
- `cleaning_log.csv`: hours dropped as closures/outages or stuck meters, and gains imputed.

`summary_metrics.csv` has one row per zone across Floors 2-7 (read by the dashboard and the web app).

- **Anomaly flags:** run an EWMA control chart on `resid` per zone; sustained positive runs are
  hours where the zone used more AC than normal for the weather and occupancy.
- **Comfort cross-check:** join those hours to PMV from the comfort analysis to label each flag
  too cool / comfortable / too warm.
- **Measuring savings later:** after a setpoint change, savings = predicted minus actual. Retrain
  monthly (as in the rolling-origin check) only on hours *before* the change, and quote the savings
  with the fractional savings uncertainty.